# The LLM Landscape: Every Major Model Family

## Evolution of LLMs

```
2017: Transformer (Attention Is All You Need)
2018: GPT-1, BERT
2019: GPT-2, XLNet, RoBERTa, DistilBERT
2020: GPT-3 (175B), T5, BART
2021: Codex, CLIP, PaLM
2022: InstructGPT, ChatGPT, Galactica, BLOOM, LLaMA-1
2023: GPT-4, LLaMA-2, Mistral, Claude-2, Gemini, Falcon, Mixtral
2024: GPT-4o, Claude-3, Gemini-1.5, LLaMA-3, Phi-3, DeepSeek-V2, Qwen2
2025: Claude-4, GPT-5, LLaMA-4, Gemini-2, DeepSeek-R1, Phi-4
```

## Model Comparison

| Model | Params | Context | Open? | Strengths |
|-------|--------|---------|-------|----------|
| GPT-4o | ~200B (est) | 128K | No | Multimodal, versatile |
| Claude 3.5 Sonnet | Unknown | 200K | No | Coding, reasoning |
| Gemini 1.5 Pro | Unknown | 1M | No | Long context |
| Llama 3.1 405B | 405B | 128K | Yes | Open, strong |
| Mistral Large | Unknown | 128K | No | European, multilingual |
| Mixtral 8x22B | 141B | 64K | Yes | MoE, efficient |
| DeepSeek-V3 | 671B | 128K | Yes | Math, code |
| Qwen2.5 72B | 72B | 128K | Yes | Multilingual |

## Mixture of Experts (MoE)

Instead of activating all parameters, MoE routes tokens to specialized "experts":

$$y = \sum_{i=1}^{n} G(x)_i \cdot E_i(x)$$

where $G(x)$ is the gating function (selects top-k experts), $E_i(x)$ is the $i$-th expert.

**Mixtral 8x7B**: 8 experts, 2 active per token. Total params: 47B, active params: ~13B per forward pass.

## Scaling Laws

Chinchilla scaling law (compute-optimal):
$$L(N, D) = \frac{A}{N^\alpha} + \frac{B}{D^\beta} + L_\infty$$

Optimal: train a model with $N$ parameters on $20N$ tokens.

In [1]:
# --- Using GPT-4o ---
from openai import OpenAI

def query_openai(prompt, model="gpt-4o-mini", system="You are helpful."):
    client = OpenAI()  # OPENAI_API_KEY from env
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

# Available OpenAI models:
openai_models = {
    "gpt-4o": "Most capable, multimodal, 128K context",
    "gpt-4o-mini": "Cheaper, faster, good for most tasks",
    "o1": "Reasoning model, thinks before answering",
    "o3-mini": "Fast reasoning model",
    "gpt-3.5-turbo": "Cheapest, fastest, older",
}

for model, desc in openai_models.items():
    print(f"{model}: {desc}")

gpt-4o: Most capable, multimodal, 128K context
gpt-4o-mini: Cheaper, faster, good for most tasks
o1: Reasoning model, thinks before answering
o3-mini: Fast reasoning model
gpt-3.5-turbo: Cheapest, fastest, older


In [2]:
# --- Using Claude ---
import anthropic

def query_claude(prompt, model="claude-sonnet-4-6", system="You are helpful."):
    client = anthropic.Anthropic()  # ANTHROPIC_API_KEY from env
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

claude_models = {
    "claude-opus-4-8": "Most capable, best reasoning",
    "claude-sonnet-4-6": "Best balance of intelligence and speed",
    "claude-haiku-4-5-20251001": "Fastest and cheapest",
}

for model, desc in claude_models.items():
    print(f"{model}: {desc}")

claude-opus-4-8: Most capable, best reasoning
claude-sonnet-4-6: Best balance of intelligence and speed
claude-haiku-4-5-20251001: Fastest and cheapest


In [3]:
# --- Using Local LLMs with Ollama ---
# Install: https://ollama.ai
# Pull: ollama pull llama3.2

import ollama

def query_ollama(prompt, model='llama3.2'):
    response = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response['message']['content']

# Popular Ollama models:
ollama_models = [
    ("llama3.2", "3B", "Meta's small Llama"),
    ("llama3.1", "8B", "Meta Llama 3.1 8B"),
    ("mistral", "7B", "Mistral 7B v0.3"),
    ("mixtral", "47B", "Mixtral 8x7B MoE"),
    ("phi4", "14B", "Microsoft Phi-4"),
    ("gemma2", "9B", "Google Gemma 2"),
    ("qwen2.5", "7B", "Alibaba Qwen 2.5"),
    ("deepseek-r1", "7B", "DeepSeek R1 distilled"),
    ("codellama", "7B", "Code generation"),
    ("nomic-embed-text", "N/A", "Embeddings model"),
]

print("Available Ollama models:")
for name, size, desc in ollama_models:
    print(f"  ollama pull {name:20s} ({size}) {desc}")

Available Ollama models:
  ollama pull llama3.2             (3B) Meta's small Llama
  ollama pull llama3.1             (8B) Meta Llama 3.1 8B
  ollama pull mistral              (7B) Mistral 7B v0.3
  ollama pull mixtral              (47B) Mixtral 8x7B MoE
  ollama pull phi4                 (14B) Microsoft Phi-4
  ollama pull gemma2               (9B) Google Gemma 2
  ollama pull qwen2.5              (7B) Alibaba Qwen 2.5
  ollama pull deepseek-r1          (7B) DeepSeek R1 distilled
  ollama pull codellama            (7B) Code generation
  ollama pull nomic-embed-text     (N/A) Embeddings model


In [4]:
# --- Using Hugging Face Transformers ---
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

def load_hf_model(model_id):
    """Load any HF model for text generation"""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    return pipeline('text-generation', model=model, tokenizer=tokenizer)

# Popular HF models:
hf_models = [
    "meta-llama/Llama-3.2-3B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "microsoft/phi-4",
    "google/gemma-2-9b-it",
    "Qwen/Qwen2.5-7B-Instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    "tiiuae/falcon-7b-instruct",
    "01-ai/Yi-1.5-9B-Chat",
]

print("Loadable HF models:")
for m in hf_models:
    print(f"  {m}")

Loadable HF models:
  meta-llama/Llama-3.2-3B-Instruct
  meta-llama/Llama-3.1-8B-Instruct
  mistralai/Mistral-7B-Instruct-v0.3
  mistralai/Mixtral-8x7B-Instruct-v0.1
  microsoft/phi-4
  google/gemma-2-9b-it
  Qwen/Qwen2.5-7B-Instruct
  deepseek-ai/DeepSeek-R1-Distill-Llama-8B
  tiiuae/falcon-7b-instruct
  01-ai/Yi-1.5-9B-Chat


In [5]:
# --- Model Selection Guide ---
model_selection = """
TASK                     RECOMMENDED MODEL
─────────────────────────────────────────────────────
General chat/reasoning   Claude Sonnet, GPT-4o
Coding                   Claude 3.5 Sonnet, DeepSeek-Coder, GPT-4o
Math/Science             o1, DeepSeek-R1, Claude Sonnet
Long documents (>100K)   Gemini 1.5 Pro, Claude 3 Opus
Cost-efficient           GPT-4o-mini, Claude Haiku, Gemini Flash
Local/private            Llama 3.1 8B, Mistral 7B (via Ollama)
Local (large, powerful)  Llama 3.1 70B, Mixtral 8x7B
Multilingual             Qwen2.5, Gemini, Claude
Image understanding      GPT-4o, Claude 3, Gemini 1.5 Pro
Embeddings               text-embedding-3-small, bge-m3, e5-large
Fine-tuning base         Llama 3.1 8B, Mistral 7B, Phi-4
"""
print(model_selection)


TASK                     RECOMMENDED MODEL
─────────────────────────────────────────────────────
General chat/reasoning   Claude Sonnet, GPT-4o
Coding                   Claude 3.5 Sonnet, DeepSeek-Coder, GPT-4o
Math/Science             o1, DeepSeek-R1, Claude Sonnet
Long documents (>100K)   Gemini 1.5 Pro, Claude 3 Opus
Cost-efficient           GPT-4o-mini, Claude Haiku, Gemini Flash
Local/private            Llama 3.1 8B, Mistral 7B (via Ollama)
Local (large, powerful)  Llama 3.1 70B, Mixtral 8x7B
Multilingual             Qwen2.5, Gemini, Claude
Image understanding      GPT-4o, Claude 3, Gemini 1.5 Pro
Embeddings               text-embedding-3-small, bge-m3, e5-large
Fine-tuning base         Llama 3.1 8B, Mistral 7B, Phi-4



## Additional Learning Resources

### Leaderboards
- [Open LLM Leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard)
- [LMSYS Chatbot Arena](https://chat.lmsys.org/)
- [Artificial Analysis](https://artificialanalysis.ai/)

### Papers
- [GPT-3 paper](https://arxiv.org/abs/2005.14165)
- [LLaMA paper](https://arxiv.org/abs/2302.13971)
- [Mistral 7B paper](https://arxiv.org/abs/2310.06825)
- [Chinchilla scaling](https://arxiv.org/abs/2203.15556)
- [DeepSeek-V3](https://arxiv.org/abs/2412.19437)
- [Phi-3](https://arxiv.org/abs/2404.14219)

### APIs & Docs
- [OpenAI API](https://platform.openai.com/docs/)
- [Anthropic API](https://docs.anthropic.com/)
- [Google AI Studio](https://ai.google.dev/)
- [Hugging Face Hub](https://huggingface.co/models)
- [Ollama](https://ollama.ai/library)